## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")    

In [57]:
from langchain_groq import ChatGroq

model=ChatGroq(model="qwen/qwen3.8-27b-chat", api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, client=<groq.resources.chat.completions.Completions object at 0x000002AA0B565550>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002AA0B486D90>, model_name='qwen/qwen3.8-27b-chat', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [58]:
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq

model = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=groq_api_key
)

model.invoke([
    HumanMessage(content="Hi, my name is Krish and I am a Chief AI Engineer.")
])

AIMessage(content='Hello Krish! It’s great to meet you. As a Chief AI Engineer, you must be working on some fascinating projects. How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says hi, introduces themselves. We should respond friendly, ask how we can help. No disallowed content. So respond accordingly.'}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 86, 'total_tokens': 157, 'completion_time': 0.148164631, 'completion_tokens_details': {'reasoning_tokens': 29}, 'prompt_time': 0.003710734, 'prompt_tokens_details': None, 'queue_time': 0.35078488, 'total_time': 0.151875365}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_60e4b492db', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0fbdf-0733-7ef3-932c-86f1aeef6fdc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 86, 'output_tokens': 71, 'total_tokens': 157, 'output_token

In [59]:
from langchain_core.messages import AIMessage

model.invoke([
    HumanMessage(content="Hi, my name is Rishabh"),
    AIMessage(content="Hi Rishabh, nice to meet you! How can I assist you today?"),
    HumanMessage(content="Hey What's my name and What do I do?")
])

AIMessage(content='Your name is **Rishabh**—you told me that in your first message.  \n\nAs for what you do, I don’t have any details yet. Could you tell me a bit about your interests, work, or what you’re looking to accomplish today? That way I can tailor my help to you.', additional_kwargs={'reasoning_content': 'We need to answer the user. The user asks: "Hey What\'s my name and What do I do?" The user earlier said "Hi, my name is Rishabh". So name is Rishabh. The user asks "What do I do?" We don\'t have info about what they do. Could ask clarifying question. Or we can respond with what we know: they introduced themselves as Rishabh, but we don\'t have info about their occupation or activities. We can ask them to share more. Should be friendly.'}, response_metadata={'token_usage': {'completion_tokens': 181, 'prompt_tokens': 116, 'total_tokens': 297, 'completion_time': 0.378314211, 'completion_tokens_details': {'reasoning_tokens': 108}, 'prompt_time': 0.029029005, 'prompt_tokens_detai


### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [60]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory  
from langchain_core.runnables.history import RunnableWithMessageHistory


store={}

'''
This function retrieves the chat message history for a given session ID.
If the session ID is not in the store, it creates a new ChatMessageHistory instance.
'''
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


## We are creating a RunnableWithMessageHistory instance that wraps the model and uses the get_session_history function to manage chat history for different sessions.
with_message_history = RunnableWithMessageHistory(model, get_session_history)

c:\Users\Rishabh Pandey\Desktop\GenAI\LANGCHAIN\venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [61]:
config = {"configurable": {"session_id": "chat1"}}

In [62]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi, my name is Rishabh")],
    config=config
)
response

AIMessage(content='Hello Rishabh! 👋 Nice to meet you. How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "Hi, my name is Rishabh". Likely a greeting. We should respond friendly, maybe ask how can we help. No disallowed content. So respond.'}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 79, 'total_tokens': 146, 'completion_time': 0.138751671, 'completion_tokens_details': {'reasoning_tokens': 39}, 'prompt_time': 0.014727878, 'prompt_tokens_details': None, 'queue_time': 0.404833266, 'total_time': 0.153479549}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_bb691ea66b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0fbdf-0d49-7af2-97b8-b7b144f13d31-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 79, 'output_tokens': 67, 'total_tokens': 146, 'output_token_details': {'reasoning': 39}})

In [63]:
response.content

'Hello Rishabh! 👋 Nice to meet you. How can I assist you today?'

In [64]:
## Change the Config-> changing the session ID

config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hey my name is Pandeyji")],
    config=config1
)
response.content


'Hello, Pandeyji! 👋 Nice to meet you. How can I assist you today?'

In [65]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey my name is Pandeyji")],
    config=config1
)
response.content

'Nice to meet you again, Pandeyji! 😊 What can I help you with today?'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [74]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder,HumanMessagePromptTemplate


# It is used to create a chat prompt template that defines the structure of the conversation. 
# The template consists of a system message that sets the context for the assistant and a placeholder for user messages. 
# in this we can give any number of messages and the model will respond accordingly.
prompt = ChatPromptTemplate(
    [
        ("system",
        "You are a helpful assistant. Answer the questions as best as you can."),
        MessagesPlaceholder(variable_name="messages"),
    ]
)
chain=prompt|model

In [75]:
chain.invoke({"messages":[HumanMessage(content="Hey my name is Pandeyji")],"language":"Hindi"})

AIMessage(content='Hello Pandeyji! Nice to meet you. How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "Hey my name is Pandeyji". Likely they are introducing themselves. The assistant should respond politely, maybe ask how can help. Keep friendly. No disallowed content.'}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 96, 'total_tokens': 162, 'completion_time': 0.139581723, 'completion_tokens_details': {'reasoning_tokens': 40}, 'prompt_time': 0.021142982, 'prompt_tokens_details': None, 'queue_time': 0.454663788, 'total_time': 0.160724705}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_803c0ba83d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0fdd9-3212-7332-bb42-d2fa614e126a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 66, 'total_tokens': 162, 'output_token_details': {'reasoning': 40}})

In [76]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)


c:\Users\Rishabh Pandey\Desktop\GenAI\LANGCHAIN\venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [80]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Rishabh")],
    config=config
)

response.content

'Nice to meet you, Rishabh! How can I help you today?'

In [81]:
## Adding some more complexity by adding multiple input variable 

prompt=ChatPromptTemplate(
    [
        ("system",
        "You are a helpful assistant. Answer the questions to the best of your ability in {language}"),
        MessagesPlaceholder(variable_name="messages")
    ]
)
chain=prompt|model

In [82]:
response=chain.invoke(
    {"messages":[HumanMessage(content="Hey my name is Pandeyji")],
    "language":"Hindi"}
)

response.content

'नमस्ते पाण्डेजी जी! आपका स्वागत है। मैं आपकी कैसे मदद कर सकता/सकती हूँ? 😊'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [83]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

c:\Users\Rishabh Pandey\Desktop\GenAI\LANGCHAIN\venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [88]:
config={"configurable":{"session_id":"chat4"}}

response=with_message_history.invoke(
    {
        'messages':[HumanMessage(content="Hey my name is Pandeyji")],
        'language':'Hindi'
    },
    config=config
)
response.content

'नमस्ते, पांडे जी! आपका स्वागत है। मैं आपकी किस प्रकार मदद कर सकता हूँ?'

In [89]:
onfig={"configurable":{"session_id":"chat5"}}

response=with_message_history.invoke(
    {
        'messages':[HumanMessage(content="What is my name?")],
        'language':'Hindi'
    },
    config=config
)
response.content

'आपका नाम पांडे जी है।'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [92]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=70, # Maximum number of tokens allowed in the message history.   
    strategy="last",  # Strategy to trim the messages. "last" means it will keep the last messages within the token limit.
    token_counter=model, # Function or model to count the number of tokens in the messages.
    include_system=True, # Whether to include system messages in the token count. If True, system messages will be counted towards the max_tokens limit.
    allow_partial=False, # Whether to allow partial tokens in the message history. If True, partial tokens will be included in the token count.
    start_on="human"   # It specifies which type of message to start trimming from. In this case, it starts trimming from the last human message
)

In [ ]:
messages=[
    SystemMessage(content="You are a helpful assistant. Answer the questions to the best of your ability."),
    AIMessage(content="Hello! How can I assist you today?"),
    HumanMessage(content="Please provide your name and occupation."),
    AIMessage(content="My name is Rishabh and I am a Chief AI Engineer."),
    HumanMessage(content="what is 2+2?"),
    AIMessage(content="2+2 is equal to 4."),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

## top 2 human message is tremmed and the rest of the messages are kept in the history. The system message is also included in the history as it is within the token limit.

c:\Users\Rishabh Pandey\Desktop\GenAI\LANGCHAIN\venv\Lib\site-packages\langchain_core\language_models\base.py:463: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))
c:\Users\Rishabh Pandey\Desktop\GenAI\LANGCHAIN\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Rishabh Pandey\Desktop\GenAI\LANGCHAIN\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Rishabh Pandey\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might requ

[SystemMessage(content='You are a helpful assistant. Answer the questions to the best of your ability.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='2+2 is equal to 4.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [99]:
## How to pass trimmer function in a chain

from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough().assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
)

response=chain.invoke(
    {"messages":messages + [HumanMessage(content="What maths probem did I ask?")],
    "language":"English"}
)

response.content

'You asked a simple addition problem: **“What is\u202f2\u202f+\u202f2?”**. It’s a basic arithmetic question that asks for the sum of the numbers 2 and 2.'

In [100]:
### Lets wrap this in the message 

with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)
config={"configurable":{"session_id":"chat6"}}

c:\Users\Rishabh Pandey\Desktop\GenAI\LANGCHAIN\venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [103]:
response=with_message_history.invoke(
    {"messages":messages + [HumanMessage(content="What maths probem did I ask?")],
    "language":"English"},
    config=config
)

response.content

'You asked a simple arithmetic question:\u202f**“What is\u202f2\u202f+\u202f4?”** (actually, it was “What is\u202f2\u202f+\u202f2?”).'